In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [2]:
chess = pd.read_csv("../data/chess_games.csv")

In [ ]:
# AQ1: Descriptive Statistics

# Create rating difference feature
chess['rating_diff'] = chess['white_rating'] - chess['black_rating']

# Basic descriptive statistics for turns
turns_stats = chess['turns'].describe()
print("Turns Statistics:")
print(turns_stats)

# Basic descriptive statistics for rating difference
rating_stats = chess['rating_diff'].describe()
print("\nRating Difference Statistics:")
print(rating_stats)

# Skewness check (important for AQ2 later)
print("\nSkewness:")
print("Turns skew:", chess['turns'].skew())
print("Rating diff skew:", chess['rating_diff'].skew())

# Plot distributions (EDA visualization)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Distribution of turns
ax[0].hist(chess['turns'], bins=50, edgecolor='white')
ax[0].set_title("Distribution of Turns")
ax[0].set_xlabel("Turns")
ax[0].set_ylabel("Frequency")

# Distribution of rating difference
ax[1].hist(chess['rating_diff'], bins=50, edgecolor='white')
ax[1].set_title("Distribution of Rating Difference")
ax[1].set_xlabel("Rating Difference")
ax[1].set_ylabel("Frequency")

fig.tight_layout()

# Save figure (IMPORTANT for grading)
plt.savefig("../images/plots/aq1_descriptive_stats.png", dpi=150, bbox_inches="tight")
plt.close()

Turns Statistics:
count    20058.000000
mean        60.465999
std         33.570585
min          1.000000
25%         37.000000
50%         55.000000
75%         79.000000
max        349.000000
Name: turns, dtype: float64

Rating Difference Statistics:
count    20058.000000
mean         7.799880
std        249.036667
min      -1605.000000
25%       -108.000000
50%          3.000000
75%        122.000000
max       1499.000000
Name: rating_diff, dtype: float64

Skewness:
Turns skew: 0.897283771438351
Rating diff skew: 0.08272337548181845


In [16]:
# AQ2: Distribution Analysis

from scipy import stats

# Create log transform
chess['log_turns'] = np.log1p(chess['turns'])

# Visual comparison
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Original distribution
ax[0].hist(chess['turns'], bins=50, edgecolor='white')
ax[0].set_title("Original Turns Distribution")
ax[0].set_xlabel("Turns")

# Log transformed distribution
ax[1].hist(chess['log_turns'], bins=50, edgecolor='white', color='orange')
ax[1].set_title("Log-Transformed Turns")
ax[1].set_xlabel("Log(Turns)")

fig.tight_layout()

plt.savefig("../images/plots/aq2_distribution.png", dpi=150, bbox_inches="tight")
plt.close()

# Normality test (sample only — important for big data)
sample = chess['turns'].sample(5000, random_state=42)

stat, p = stats.shapiro(sample)

print("Shapiro test p-value:", p)

# Skewness comparison
print("Original skew:", chess['turns'].skew())
print("Log skew:", chess['log_turns'].skew())

Shapiro test p-value: 2.2525033182761157e-34
Original skew: 0.897283771438351
Log skew: -1.3708338858711593


## AQ2 Interpretation
The log transformation reduces skewness and improves the symmetry of the distribution, making the data more suitable for statistical modeling and parametric tests. However, the data still does not perfectly follow a normal distribution, which is expected in real-world behavioral datasets like chess games.

In [10]:
# AQ3: Correlation Matrix

# Keep only numeric columns
numeric_df = chess.select_dtypes(include=np.number)

# Correlation matrix
corr = numeric_df.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(
    corr,
    cmap='RdYlBu_r',
    center=0,
    annot=False,
    ax=ax
)

ax.set_title("Chess Dataset Correlation Matrix")

fig.tight_layout()

# Save figure
plt.savefig("../images/plots/aq3_correlation.png", dpi=150, bbox_inches="tight")
plt.close()

## AQ3 Interpretation

The correlation matrix shows clear relationships between key variables in the chess dataset. The strongest relationship exists between white_rating and black_rating, indicating that the matchmaking system tends to pair players of similar skill levels. In addition, rating_diff is strongly related to both white_rating and black_rating, which is expected since it is a direct mathematical combination of the two variables.

On the other hand, there is a very weak correlation between game length (turns and log_turns) and player rating variables. This suggests that player skill level alone does not have a strong linear relationship with how long a game lasts.

A key potential confounding factor in this dataset is the opening strategy (opening moves or opening family). Certain openings can systematically lead to shorter or longer games regardless of player rating. Therefore, the opening choice may influence both the perceived strength of players and the game duration, which can weaken or mask true relationships between rating and game length.

Overall, the analysis indicates that structural factors such as matchmaking have stronger linear relationships, while game duration is influenced by more complex and non-linear factors.

In [17]:
# AQ4: Chi-Squared Test

from scipy.stats import chi2_contingency

# Create binary column: white wins or not
chess['white_win'] = (chess['winner'] == 'White')

# Create rating groups (example: low/high)
chess['rating_group'] = pd.cut(
    chess['white_rating'],
    bins=[0, 1400, 1800, 3000],
    labels=['Low', 'Medium', 'High']
)

# Contingency table
contingency = pd.crosstab(chess['rating_group'], chess['white_win'])

print("Contingency Table:\n", contingency)

# Chi-square test
chi2, p, dof, expected = chi2_contingency(contingency)

print("\nChi-square test results:")
print("Chi2:", chi2)
print("p-value:", p)

# Effect size (Cramer's V)
n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print("\nEffect size (Cramer's V):", cramers_v)

Contingency Table:
 white_win     False  True 
rating_group              
Low            2993   2102
Medium         4935   5173
High           2129   2726

Chi-square test results:
Chi2: 234.6757730305165
p-value: 1.0985083646650612e-51

Effect size (Cramer's V): 0.10816588676773789


## AQ4 Interpretation

There is a statistically significant relationship between player rating group and White win rate (χ² = 234.68, p < 0.001). However, the effect size is small (Cramér’s V ≈ 0.11), indicating that the relationship is weak in practical terms. This highlights that statistical significance alone can be misleading when sample sizes are large.

In [15]:
# AQ5: Confidence Intervals

from scipy.stats import t

# Split data
rated = chess[chess['rated'] == True]['turns']
unrated = chess[chess['rated'] == False]['turns']

# 1. Parametric CI function

def confidence_interval(data, confidence=0.95):
    n = len(data)
    mean = data.mean()
    se = stats.sem(data)  # standard error
    h = se * t.ppf((1 + confidence) / 2., n - 1)
    return mean - h, mean + h

# Compute CIs
rated_ci = confidence_interval(rated)
unrated_ci = confidence_interval(unrated)

print("Rated 95% CI:", rated_ci)
print("Unrated 95% CI:", unrated_ci)

# 2. Bootstrap CI

def bootstrap_ci(data, n_boot=1000, confidence=0.95):
    rng = np.random.default_rng(42)
    means = []

    for _ in range(n_boot):
        sample = rng.choice(data, size=len(data), replace=True)
        means.append(sample.mean())

    lower = np.percentile(means, (1 - confidence) / 2 * 100)
    upper = np.percentile(means, (1 + confidence) / 2 * 100)

    return lower, upper

rated_boot = bootstrap_ci(rated.values)
unrated_boot = bootstrap_ci(unrated.values)

print("Rated Bootstrap CI:", rated_boot)
print("Unrated Bootstrap CI:", unrated_boot)

Rated 95% CI: (np.float64(61.44233966617739), np.float64(62.48276092187585))
Unrated 95% CI: (np.float64(53.26225453567002), np.float64(55.28091738336662))
Rated Bootstrap CI: (np.float64(61.432373878056325), np.float64(62.499755493655215))
Unrated Bootstrap CI: (np.float64(53.204701511657696), np.float64(55.29671406610299))


## AQ5 Interpretation
Rated games last significantly longer than unrated games. The 95% confidence intervals for the two groups do not overlap, indicating a clear and statistically reliable difference in mean game length. Both parametric and bootstrap confidence intervals produce consistent results, confirming robustness of the finding.